In [1]:
import pandas as pd 
import os 
import numpy as np 

In [2]:
chunks = [] 
for chunk in pd.read_csv("bill_schedule.csv", chunksize = 500_000):
    chunk['io_time'] = pd.to_datetime(chunk['io_time']) 
    chunks.append(chunk) 
sche = pd.concat(chunks, ignore_index = True) 
del chunk 
wh_1a = pd.read_csv('warehouse_1A.csv')
set_1a = set(wh_1a['name']) 

sche['next_wh'] = sche.groupby('bill_code')['warehouse_name'].shift(1) 
sche['pre_wh'] = sche.groupby('bill_code')['warehouse_name'].shift(-1) 
sche_first = sche[sche['warehouse_name'] != sche['next_wh']].copy() 
sche_last = sche[sche['warehouse_name'] != sche['pre_wh']].copy() 

sche_first['wh_rank'] = sche_first.groupby('bill_code').cumcount() 
sche_last['wh_rank'] = sche_last.groupby('bill_code').cumcount() 
sche_1a = sche_first[sche_first['warehouse_name'].isin(set_1a)] 
count_1a = sche_1a.groupby('bill_code').size().reset_index(name = 'count_1a') 
count_1a = count_1a[count_1a['count_1a'] >= 2] 
valid_bills = set(count_1a['bill_code']) 

sche_first = sche_first[sche_first['bill_code'].isin(valid_bills)] 
max_rank_first = sche_first.groupby('bill_code')['wh_rank'].transform('max') 
sche_first['wh_rank_rev'] = max_rank_first - sche_first['wh_rank'] 

sche_last = sche_last[sche_last['bill_code'].isin(valid_bills)] 
max_rank_last = sche_last.groupby('bill_code')['wh_rank'].transform('max') 
sche_last['wh_rank_rev'] = max_rank_last - sche_last['wh_rank'] 

sche_first_1a = sche_first[(sche_first['wh_rank'] > 0) & (sche_first['warehouse_name'].isin(set_1a))] 
wh_o1a_first = sche_first_1a.drop_duplicates(subset=['bill_code'], keep='first') 
wh_o1a = (wh_o1a_first[['bill_code', 'warehouse_name', 'io_time', 'io_status']]
          .rename(columns={'warehouse_name': 'kho_o1a', 'io_time': 'time_o1a', 'io_status': 'status_o1a'})) 

In [3]:
data = wh_o1a['kho_o1a'].value_counts() 
import plotly.express as px
fig = px.bar(data) 
fig.show()

In [ ]:
# -*- coding: utf-8 -*-
"""
arrival_forecaster.py — Core ML Model Class (Độc lập nguồn dữ liệu)
"""

import math
import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMRegressor
from lightgbm.callback import early_stopping
from sklearn.metrics import mean_absolute_error, median_absolute_error, r2_score


class ArrivalForecaster:
    """Mô hình dự báo thời gian hàng đến kho 1A, độc lập với nguồn dữ liệu."""

    FEATURE_COLS = [
        "warehouse_source", "io_status_source", "io_time_source", "snapshot_hour_numeric",
        "hours_since_last_scan", "stage_depth", "VD_type", "origin_province", "origin_district",
        "destination_province", "destination_district", "service", "estimated_delivery_days",
        "estimated_delivery_days_cat", "creation_delivery_dow_diff", "creation_delivery_dow_diff_circular",
        "same_creation_delivery_dow", "same_creation_delivery_month", "creation_hour_sin",
        "creation_hour_cos", "creation_dow_sin", "creation_dow_cos", "creation_month_sin",
        "creation_month_cos", "province_route", "district_route", "actual_weight",
        "log_actual_weight", "weight_bin", "service_VD_type"
    ]

    CAT_COLS = [
        "warehouse_source", "io_status_source", "VD_type", "origin_province", "origin_district",
        "destination_province", "destination_district", "service", "estimated_delivery_days_cat",
        "province_route", "district_route", "weight_bin", "service_VD_type"
    ]

    def __init__(self, target_warehouse: str = None):
        self.target_warehouse = target_warehouse
        self.bundle = {}
        self.is_fitted = False

    # ─────────────────────────────────────────────────────────────
    # FIT: CHỈ NHẬN VÀO 1 DATAFRAME ĐÃ CHUẨN HÓA
    # ─────────────────────────────────────────────────────────────
    def fit(
        self,
        train_df: pd.DataFrame,
        target_warehouse: str = None,
        train_split: float = 0.80,
        min_samples_multimodal: int = 200,
    ) -> "ArrivalForecaster":
        """
        Huấn luyện mô hình từ DataFrame chuẩn.
        train_df bắt buộc có: 'bill_code', 'warehouse_source', 'io_status_source', 'arrival_offset_hours'
        """
        if target_warehouse:
            self.target_warehouse = target_warehouse
        dest = self.target_warehouse or "Global_1A"

        # Kiểm tra cột bắt buộc
        req_cols = ["bill_code", "warehouse_source", "io_status_source", "arrival_offset_hours"]
        for c in req_cols:
            if c not in train_df.columns:
                raise ValueError(f"Thiếu cột bắt buộc trong train_df: '{c}'")

        df = train_df.dropna(subset=["arrival_offset_hours"]).copy()
        
        # Bổ sung các feature thiếu thành NaN
        for col in self.FEATURE_COLS:
            if col not in df.columns:
                df[col] = np.nan

        # 1. Chia Train / Val / Test theo bill_code
        bills = df["bill_code"].unique()
        rng = np.random.default_rng(42)
        rng.shuffle(bills)
        n_b = len(bills)
        tr_b, va_b, te_b = set(bills[:int(n_b * train_split)]), set(bills[int(n_b * train_split):int(n_b * (train_split + 0.1))]), set(bills[int(n_b * (train_split + 0.1)):])

        train = df[df["bill_code"].isin(tr_b)].copy()
        val = df[df["bill_code"].isin(va_b)].copy()
        test = df[df["bill_code"].isin(te_b)].copy()

        # 2. Outlier Filtering (Fit IQR trên Train)
        grp = train.groupby(["warehouse_source", "io_status_source"])["arrival_offset_hours"]
        q1, q3 = grp.quantile(0.25), grp.quantile(0.75)
        iqr = q3 - q1
        lo_bound, up_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr

        def apply_outlier(d):
            k = list(zip(d["warehouse_source"], d["io_status_source"]))
            lo = pd.Series(k, index=d.index).map(lo_bound).fillna(0.0)
            hi = pd.Series(k, index=d.index).map(up_bound).fillna(72.0)
            return d[(d["arrival_offset_hours"] >= lo) & (d["arrival_offset_hours"] <= hi) & (d["arrival_offset_hours"] >= 0)].copy()

        train, val, test = apply_outlier(train), apply_outlier(val), apply_outlier(test)

        # 3. Categorical Encoding Mapping
        cat_mappings = {}
        for c in self.CAT_COLS:
            train[c] = train[c].astype("category")
            cat_mappings[c] = train[c].cat.categories.tolist()
            val[c] = pd.Categorical(val[c], categories=train[c].cat.categories)
            test[c] = pd.Categorical(test[c], categories=train[c].cat.categories)

        # 4. Huấn luyện Segment Models
        segment_models = {}
        segments = train["warehouse_source"].dropna().unique().tolist()
        bill_cnts = train.groupby("bill_code").size()
        train["sample_weight"] = 1.0 / np.sqrt(train["bill_code"].map(bill_cnts))

        def get_lgbm_params(n):
            n = max(int(n), 1)
            return {
                "objective": "regression", "n_estimators": 300 if n < 5000 else 800,
                "learning_rate": 0.05 if n < 5000 else 0.03, "num_leaves": max(15, min(63, int(n / 100))),
                "max_depth": max(4, min(10, int(math.log2(max(n, 2))))), "min_child_samples": max(1, min(50, int(n * 0.02))),
                "subsample": 0.8, "colsample_bytree": 0.8, "random_state": 42, "n_jobs": -1, "verbose": -1
            }

        for seg in segments:
            tr_m, va_m = train["warehouse_source"] == seg, val["warehouse_source"] == seg
            X_tr, y_tr, w_tr = train[tr_m][self.FEATURE_COLS], train[tr_m]["arrival_offset_hours"].values, train[tr_m]["sample_weight"].values
            X_va, y_va = val[va_m][self.FEATURE_COLS], val[va_m]["arrival_offset_hours"].values
            n_samples = len(X_tr)
            if n_samples < 5:
                continue

            n_clusters = 1
            if n_samples >= min_samples_multimodal:
                y_sub = y_tr if n_samples <= 10_000 else y_tr[np.random.default_rng(42).choice(n_samples, 10_000, replace=False)]
                db = DBSCAN(eps=0.5, min_samples=max(10, int(len(y_sub) * 0.03))).fit(y_sub.reshape(-1, 1))
                if (db.labels_ != -1).any():
                    means = pd.Series(y_sub[db.labels_ != -1]).groupby(db.labels_[db.labels_ != -1]).mean().sort_values().values
                    if len(means) >= 2:
                        c_tr = np.argmin(np.abs(y_tr[:, None] - means[None, :]), axis=1)
                        n_clusters = len(means)

            if n_clusters >= 2:
                X_tr_clf = X_tr.copy()
                for c in self.CAT_COLS:
                    X_tr_clf[c] = X_tr_clf[c].cat.codes if hasattr(X_tr_clf[c], "cat") else pd.Categorical(X_tr_clf[c]).codes
                clf = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1).fit(X_tr_clf.fillna(0), c_tr)
                regressors = {}
                for cid in np.unique(c_tr):
                    mc = c_tr == cid
                    Xc, yc, wc = X_tr[mc], y_tr[mc], w_tr[mc]
                    reg = LGBMRegressor(**get_lgbm_params(len(Xc)))
                    if len(Xc) >= 50:
                        cut = int(len(Xc) * 0.85)
                        reg.fit(Xc.iloc[:cut], yc[:cut], sample_weight=wc[:cut], eval_set=[(Xc.iloc[cut:], yc[cut:])], callbacks=[early_stopping(30, verbose=False)], categorical_feature=self.CAT_COLS)
                    else:
                        reg.fit(Xc, yc, sample_weight=wc, categorical_feature=self.CAT_COLS)
                    regressors[cid] = reg
                segment_models[seg] = {"type": "multi", "classifier": clf, "regressors": regressors}
            else:
                reg = LGBMRegressor(**get_lgbm_params(len(X_tr)))
                kw = {"sample_weight": w_tr, "categorical_feature": self.CAT_COLS}
                if len(X_va) > 0:
                    kw["eval_set"], kw["callbacks"] = [(X_va, y_va)], [early_stopping(50, verbose=False)]
                reg.fit(X_tr, y_tr, **kw)
                segment_models[seg] = {"type": "single", "regressor": reg}

        # Fallback OTHER
        m_other = LGBMRegressor(**get_lgbm_params(len(train)))
        kw_o = {"sample_weight": train["sample_weight"].values, "categorical_feature": self.CAT_COLS}
        if len(val) > 0:
            kw_o["eval_set"], kw_o["callbacks"] = [(val[self.FEATURE_COLS], val["arrival_offset_hours"].values)], [early_stopping(50, verbose=False)]
        m_other.fit(train[self.FEATURE_COLS], train["arrival_offset_hours"].values, **kw_o)
        segment_models["OTHER"] = {"type": "single", "regressor": m_other}

        # Evaluate Metrics
        y_test, X_test = test["arrival_offset_hours"].values, test[self.FEATURE_COLS]
        test_preds = np.zeros(len(test))
        for seg in test["warehouse_source"].unique():
            m = test["warehouse_source"] == seg
            if m.sum() > 0:
                mi = segment_models.get(seg, segment_models["OTHER"])
                Xt = X_test[m]
                if mi["type"] == "multi":
                    Xt_c = Xt.copy()
                    for c in self.CAT_COLS:
                        Xt_c[c] = Xt_c[c].cat.codes if hasattr(Xt_c[c], "cat") else pd.Categorical(Xt_c[c]).codes
                    cp = mi["classifier"].predict(Xt_c.fillna(0))
                    yp = np.zeros(len(Xt))
                    for cid in np.unique(cp):
                        idx = np.where(cp == cid)[0]
                        yp[idx] = mi["regressors"][cid].predict(Xt.iloc[idx]) if cid in mi["regressors"] else np.mean([mi["regressors"][k].predict(Xt.iloc[idx]) for k in mi["regressors"]], axis=0)
                    test_preds[m] = yp
                else:
                    test_preds[m] = mi["regressor"].predict(Xt)

        metrics = {
            "test_mae": float(mean_absolute_error(y_test, test_preds)) if len(y_test) else np.nan,
            "test_medae": float(median_absolute_error(y_test, test_preds)) if len(y_test) else np.nan,
            "test_r2": float(r2_score(y_test, test_preds)) if len(y_test) else np.nan,
        }

        self.bundle = {
            "warehouse_name": dest,
            "segment_models": segment_models,
            "category_mappings": cat_mappings,
            "outlier_bounds": {"lower": lo_bound, "upper": up_bound},
            "train_metrics": metrics
        }
        self.is_fitted = True
        print(f"[{dest}] Hoàn tất fit! MAE: {metrics['test_mae']:.2f}h | MedAE: {metrics['test_medae']:.2f}h | R²: {metrics['test_r2']:.4f}")
        return self

    # ─────────────────────────────────────────────────────────────
    # PREDICT: CHỈ NHẬN VÀO 1 DATAFRAME / DICT CHUẨN
    # ─────────────────────────────────────────────────────────────
    def predict(self, df: pd.DataFrame) -> pd.DataFrame:
        """Dự báo cho DataFrame chuẩn."""
        if not self.bundle:
            raise ValueError("Mô hình chưa được fit hoặc load!")

        result = df.copy()
        if "bill_code" not in result.columns:
            result["bill_code"] = [f"bill_{i}" for i in range(len(result))]

        raw_wh = result.get("warehouse_name", result.get("warehouse_source", pd.Series(["UNKNOWN"] * len(result)))).astype(str)
        raw_st = result.get("io_status", result.get("io_status_source", pd.Series(["UNKNOWN"] * len(result)))).astype(str)
        result["warehouse_source"] = raw_wh
        result["io_status_source"] = raw_st

        # Bổ sung các cột thiếu & categorical mapping
        cat_mappings = self.bundle["category_mappings"]
        for c in self.FEATURE_COLS:
            if c not in result.columns:
                result[c] = np.nan

        for c, cats in cat_mappings.items():
            if c in result.columns:
                result[c] = pd.Categorical(result[c], categories=cats)

        # Inference
        X = result[self.FEATURE_COLS]
        preds = np.zeros(len(X))
        seg_models = self.bundle["segment_models"]

        routing = raw_wh.fillna("OTHER").values
        for src in np.unique(routing):
            m = routing == src
            if m.sum() > 0:
                mi = seg_models.get(src, seg_models.get("OTHER"))
                Xt = X[m]
                if mi and mi["type"] == "multi":
                    Xt_c = Xt.copy()
                    for c in self.CAT_COLS:
                        Xt_c[c] = Xt_c[c].cat.codes if hasattr(Xt_c[c], "cat") else pd.Categorical(Xt_c[c]).codes
                    cp = mi["classifier"].predict(Xt_c.fillna(0))
                    yp = np.zeros(len(Xt))
                    for cid in np.unique(cp):
                        idx = np.where(cp == cid)[0]
                        yp[idx] = mi["regressors"][cid].predict(Xt.iloc[idx]) if cid in mi["regressors"] else np.mean([mi["regressors"][k].predict(Xt.iloc[idx]) for k in mi["regressors"]], axis=0)
                    preds[m] = yp
                elif mi:
                    preds[m] = mi["regressor"].predict(Xt)

        result["predicted_arrival_offset_hours"] = np.clip(preds, 0.0, None)
        
        # Nếu có io_time thì tính thời gian đến datetime
        if "io_time" in result.columns:
            io_t = pd.to_datetime(result["io_time"], errors="coerce")
            result["predicted_arrival_time"] = io_t + pd.to_timedelta(result["predicted_arrival_offset_hours"], unit="h")
        else:
            result["predicted_arrival_time"] = None

        return result[["bill_code", "warehouse_source", "io_status_source", "predicted_arrival_offset_hours", "predicted_arrival_time"]].copy()

    def predict_one(self, bill_info: dict) -> dict:
        """Dự báo cho 1 bản ghi dạng dictionary."""
        df = pd.DataFrame([bill_info])
        return self.predict(df).iloc[0].to_dict()

    def save(self, filepath: str):
        """Lưu model ra file .pkl."""
        joblib.dump(self.bundle, filepath, compress=3)

    @classmethod
    def load(cls, filepath: str) -> "ArrivalForecaster":
        """Nạp model từ file .pkl."""
        bundle = joblib.load(filepath)
        inst = cls(target_warehouse=bundle.get("warehouse_name"))
        inst.bundle = bundle
        inst.is_fitted = True
        return inst


In [ ]:
# -*- coding: utf-8 -*-
"""
preprocessor.py — Hàm tiện ích trích xuất dataset huấn luyện từ 3 file thô.
"""
import pandas as pd
import numpy as np


def prepare_training_dataset(
    bill_schedule_df: pd.DataFrame,
    bill_df: pd.DataFrame,
    warehouse_1a_df: pd.DataFrame,
    target_warehouse: str,
    snapshot_hour: int = 20
) -> pd.DataFrame:
    """
    Biến đổi 3 bảng thô thành 1 DataFrame chuẩn có cột nhãn 'arrival_offset_hours'.
    """
    dest = target_warehouse
    set_1a = set(warehouse_1a_df["name"].dropna().str.strip())
    
    sche = bill_schedule_df.copy()
    sche["io_time"] = pd.to_datetime(sche["io_time"])
    sche = sche.sort_values(["bill_code", "io_time"]).reset_index(drop=True)

    # 1. Lọc bills đến kho đích
    arrival_mask = (sche["warehouse_name"] == dest) & (sche["io_status"] == "IN")
    valid_bills = sche.loc[arrival_mask, "bill_code"].unique()
    sche = sche[sche["bill_code"].isin(valid_bills)].copy()
    sche["row_id"] = sche.groupby("bill_code").cumcount()
    
    arrival_rows = sche[(sche["warehouse_name"] == dest) & (sche["io_status"] == "IN")]
    first_selected = arrival_rows.groupby("bill_code")["row_id"].min().rename("first_row")
    sche = sche.merge(first_selected, on="bill_code", how="inner")
    sche_filtered = sche[sche["row_id"] <= sche["first_row"]].drop(columns=["row_id", "first_row"])

    # 2. Loại bỏ bills qua 1A khác trước đó
    other_1a = set_1a - {dest}
    bad_bills = sche_filtered.loc[sche_filtered["warehouse_name"].isin(other_1a), "bill_code"].unique()
    sche_filtered = sche_filtered[~sche_filtered["bill_code"].isin(bad_bills)]

    # 3. Tạo chặng stages
    traj = sche_filtered.groupby("bill_code").agg({"warehouse_name": list, "io_status": list, "io_time": list}).reset_index()
    arrival_map = arrival_rows.groupby("bill_code")["io_time"].first()
    traj["arrival_time"] = traj["bill_code"].map(arrival_map)
    traj = traj[traj["io_status"].apply(len) >= 2].copy()

    traj["warehouse_name"] = traj["warehouse_name"].str[:-1]
    traj["io_status"] = traj["io_status"].str[:-1]
    traj["io_time"] = traj["io_time"].str[:-1]
    stages = traj.explode(["warehouse_name", "io_status", "io_time"]).dropna(subset=["io_status", "warehouse_name", "io_time"])
    stages["stage_depth"] = stages.groupby("bill_code").cumcount()
    stages = stages.rename(columns={"warehouse_name": "warehouse_source", "io_status": "io_status_source"})
    stages["io_time_source_datetime"] = pd.to_datetime(stages["io_time"])
    t = stages["io_time_source_datetime"]
    stages["io_time_source"] = t.dt.hour + t.dt.minute / 60.0 + t.dt.second / 3600.0

    # 4. Merge bill features & Snapshots
    df = bill_df.merge(stages, on="bill_code", how="inner")
    
    # Grid Snapshot 20h
    scan_points = df[["bill_code", "io_time_source_datetime"]].rename(columns={"io_time_source_datetime": "snapshot_time"})
    bounds = df.groupby("bill_code").agg(start_time=("io_time_source_datetime", "min"), arrival_time=("arrival_time", "first")).reset_index()
    bounds["days"] = (bounds["arrival_time"].dt.normalize() - bounds["start_time"].dt.normalize()).dt.days
    max_d = int(bounds["days"].max()) if not bounds.empty and bounds["days"].max() > 0 else 0
    snap_dfs = []
    for d in range(max_d + 1):
        sub = bounds[bounds["days"] >= d].copy()
        sub["snapshot_time"] = sub["start_time"].dt.normalize() + pd.Timedelta(days=d, hours=snapshot_hour)
        sub = sub[(sub["snapshot_time"] >= sub["start_time"]) & (sub["snapshot_time"] < sub["arrival_time"])]
        if not sub.empty:
            snap_dfs.append(sub[["bill_code", "snapshot_time"]])
    snap_df_20h = pd.concat(snap_dfs, ignore_index=True) if snap_dfs else pd.DataFrame(columns=["bill_code", "snapshot_time"])
    grid_df = pd.concat([scan_points, snap_df_20h], ignore_index=True).drop_duplicates(subset=["bill_code", "snapshot_time"]).sort_values("snapshot_time")

    df = pd.merge_asof(grid_df, df, left_on="snapshot_time", right_on="io_time_source_datetime", by="bill_code", direction="backward")
    df = df.dropna(subset=["io_time_source_datetime", "arrival_time"]).copy()
    
    # Tính nhãn target
    df["arrival_offset_hours"] = (df["arrival_time"] - df["snapshot_time"]).dt.total_seconds() / 3600.0
    df["hours_since_last_scan"] = (df["snapshot_time"] - df["io_time_source_datetime"]).dt.total_seconds() / 3600.0
    st = df["snapshot_time"]
    df["snapshot_hour_numeric"] = st.dt.hour + st.dt.minute / 60.0 + st.dt.second / 3600.0

    return df
